In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.shape
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

# Titanic Dataset Analysis

This notebook performs data cleaning, feature engineering, and feature selection on the Titanic dataset.

The goal is to prepare the data for predictive modelling of passenger survival.

We import necessary libraries for data manipulation and visualization.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

We load the Titanic dataset from Google Drive and display the first few rows to understand its structure.

In [5]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


We inspect the dataset to check column types and identify missing values that need to be handled.

In [6]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


# Data Cleaning

In this section, we handle missing values, remove unnecessary columns, and ensure the dataset is consistent for analysis.

Proper cleaning improves the quality of feature engineering and model performance.

We first identify missing values in each column to decide the best strategy for handling them.

In [7]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


We fill missing values in Age using the median and Embarked using the mode.
Cabin column is dropped because it contains too many missing values to be useful.

In [8]:
# Age → fill with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Embarked → fill with mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Cabin → too many missing values, so we drop it
df.drop(columns=['Cabin'], inplace=True)

/tmp/ipykernel_23751/3042016390.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_23751/3042016390.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [9]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


We remove duplicate rows to ensure data consistency and avoid bias in analysis.

In [10]:
df.drop_duplicates(inplace=True)

# Feature Engineering

In this section, we create new meaningful features from existing data to improve model performance.

These features help capture hidden patterns such as family size, social status, and age grouping.

We create FamilySize by combining siblings/spouses and parents/children.
We also create IsAlone to indicate whether a passenger travelled alone.

In [11]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

We extract titles (Mr, Mrs, Miss, etc.) from passenger names to capture social status information.

In [12]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_23751/172848211.py:1: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


We group passengers into age categories to simplify age-based patterns in survival rates.

In [13]:
df['AgeGroup'] = pd.cut(df['Age'],
                        bins=[0,12,20,60,100],
                        labels=['Child','Teen','Adult','Senior'])

We calculate fare per person to better understand ticket pricing fairness across families.

In [14]:
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

We convert categorical variables into numerical format using one-hot encoding to prepare data for machine learning models.

In [15]:
df = pd.get_dummies(df, columns=['Sex','Embarked','Title','AgeGroup'], drop_first=True)

# Feature Selection

In this section, we identify the most important features that influence survival.

We use correlation analysis and a Random Forest model to rank feature importance.

We separate the dataset into features (X) and target variable (y).
The target is whether a passenger survived.

In [16]:
from sklearn.ensemble import RandomForestClassifier

# Separate features and target
X = df.drop(columns=['Survived','Name','Ticket'])
y = df['Survived']

We train a Random Forest model to evaluate the importance of each feature in predicting survival.

In [17]:
model = RandomForestClassifier(random_state=42)
model.fit(X, y)

RandomForestClassifier(random_state=42)

The most important features influencing survival are ranked using a Random Forest model.
Top features include passenger class, sex, fare, and engineered features such as family size and title.

In [18]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)
importance.sort_values(ascending=False).head(10)


,0
PassengerId,0.144499
Title_Mr,0.130701
Fare,0.127946
FarePerPerson,0.122249
Age,0.116203
Sex_male,0.100808
Pclass,0.053167
FamilySize,0.037042
Title_Mrs,0.029886
SibSp,0.027414


In [19]:
df.to_csv('train_cleaned.csv', index=False)

##  Final Conclusion

This project analyzed the Titanic dataset with the goal of preparing data for survival prediction through structured data preprocessing, feature engineering, and feature selection.

###  Data Cleaning

The dataset was first explored to identify missing values, data inconsistencies, and irrelevant features. Missing values in **Age** were filled using the median, while **Embarked** was filled using the mode. The **Cabin** column was removed due to excessive missing values. Duplicate records were also eliminated to ensure data integrity.

###  Feature Engineering

New meaningful features were created to improve model performance and capture hidden patterns in the data. These included:

* **FamilySize** (SibSp + Parch + 1) to represent group travel patterns
* **IsAlone** to identify solo travelers
* **Title** extracted from passenger names to capture social status
* **AgeGroup** to categorize passengers into life stages
* **FarePerPerson** to normalize ticket cost per individual

Categorical variables were encoded using one-hot encoding, and skewed numerical features such as Fare were transformed using logarithmic scaling to improve distribution quality.

### Feature Selection

A Random Forest model was used to evaluate feature importance. The most influential variables in predicting survival included **Sex, Pclass, Fare, Age, FamilySize, and Title-based features**, showing that both socio-economic status and engineered features significantly impact survival likelihood.

###  Key Insights

* Gender and passenger class were the strongest predictors of survival
* Passengers traveling alone had lower survival chances
* Higher fares and higher social titles were associated with increased survival probability
* Feature engineering significantly improved data interpretability

### Conclusion

The analysis demonstrates that careful data cleaning and thoughtful feature engineering greatly enhance the quality of a dataset. The transformed dataset is now well-structured and suitable for training machine learning models for survival prediction
